# 02b — Intrinsic vs Ambient Dimension

## The One-Sentence Version
The space your data lives *in* is not the space your data moves *through*.

## What You'll Build Intuition For
- The difference between ambient dimension (number of columns) and intrinsic dimension (true degrees of freedom)
- How low-dimensional structure hides inside high-dimensional data
- How to estimate intrinsic dimension from data

## Prerequisites
- [02a — Correlation and Redundancy](02a_correlation_and_redundancy.ipynb) — spotting redundancy in features

## The Story

Imagine a sheet of paper on a table. The paper is a 2D object — you can describe any point on it with two numbers (x, y position). But the paper sits in a 3D room. If you picked up the paper and tilted it, you'd need three numbers (x, y, z) to describe the same points — even though the paper itself is still flat.

The paper has **intrinsic dimension 2** (it's a flat surface) but **ambient dimension 3** (the room it sits in). Dimensionality reduction is the act of looking at the paper in 3D and realising: "Wait, this is really just 2D."

Now scale that up. Your dataset has 500 columns — ambient dimension 500. But maybe the data actually varies along only 8 independent directions. The intrinsic dimension is 8. The other 492 columns are correlated echoes, noise, and redundancy.

This notebook builds the intuition for that gap, and shows you how to measure it.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.decomposition import PCA

from utils.plotting import apply_style, COLOURS
from utils.data_generators import make_low_d_in_high_d

apply_style()

## A Sheet of Paper in a Room

Let's generate a 2D flat surface and embed it in 3D space with a random rotation and offset. The data is *intrinsically* 2D — but it *lives* in 3D.

In [ ]:
# Generate a 2D plane embedded in 3D
n = 500
rng = np.random.default_rng(42)

# 2D intrinsic coordinates
u = rng.uniform(-1, 1, n)
v = rng.uniform(-1, 1, n)

# Embed in 3D: random rotation + offset + tiny noise
rotation = rng.standard_normal((2, 3))
intrinsic = np.column_stack([u, v])
X_3d = intrinsic @ rotation + np.array([5, 3, 7]) + rng.standard_normal((n, 3)) * 0.02

# Interactive 3D scatter
fig_3d = go.Figure(data=[go.Scatter3d(
    x=X_3d[:, 0], y=X_3d[:, 1], z=X_3d[:, 2],
    mode="markers",
    marker=dict(size=2, color=u, colorscale="Viridis", opacity=0.7),
)])
fig_3d.update_layout(
    title="A flat 2D surface embedded in 3D — rotate to see it's flat",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    width=700, height=500, margin=dict(l=0, r=0, t=40, b=0),
)
fig_3d.show()

print(f"Data shape: {X_3d.shape} — lives in 3D")
print(f"Intrinsic dimension: 2 — it's a flat sheet")

In [ ]:
# PCA confirms: 2 components capture (almost) everything
pca_3d = PCA().fit(X_3d)
print("Explained variance per component:")
for i, ev in enumerate(pca_3d.explained_variance_ratio_):
    print(f"  PC{i+1}: {ev:.4f} ({ev:.1%})")

print(f"\n2 components capture {sum(pca_3d.explained_variance_ratio_[:2]):.2%} of variance")
print("The 3rd component is just noise — the data is truly 2D.")

## A Curve in 2D

A sine wave lives in a 2D plane (x, y axes). But the data only varies along *one* direction — position along the curve. Intrinsic dimension = 1, ambient dimension = 2.

In [ ]:
# A sine wave: 1D intrinsic, 2D ambient
t = np.linspace(0, 4 * np.pi, 300)
x_curve = t + rng.standard_normal(300) * 0.1
y_curve = np.sin(t) + rng.standard_normal(300) * 0.1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 2D view
ax1.scatter(x_curve, y_curve, s=10, alpha=0.6, c=t, cmap="viridis")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_title("Ambient view: looks 2D")

# 1D view — position along the curve
jitter = rng.uniform(-0.1, 0.1, 300)
ax2.scatter(t, jitter, s=10, alpha=0.6, c=t, cmap="viridis")
ax2.set_xlabel("Position along curve (t)")
ax2.set_yticks([])
ax2.set_ylim(-0.5, 0.5)
ax2.set_title("Intrinsic view: really just 1D")

fig.suptitle("Sine wave: intrinsic d=1, ambient d=2", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/02b_curve_in_2d.png", dpi=150, bbox_inches="tight")
plt.show()

print("One number (t = position along the curve) captures all the information.")
print("The second dimension (y) is fully determined by t — it's redundant.")

## Higher-Dimensional Embeddings

The same principle works at scale. Let's generate data that truly lives in 3 dimensions, but embed it in 100D via a random projection plus noise.

In [ ]:
# 3D data embedded in 100D
X_100d, Z_true = make_low_d_in_high_d(n=500, intrinsic_d=3, ambient_d=100, seed=42)

print(f"Data shape: {X_100d.shape} — looks like 100 dimensions")
print(f"True intrinsic dimension: {Z_true.shape[1]}")

# PCA reveals the truth
pca_100d = PCA(random_state=42).fit(X_100d)
cumvar = np.cumsum(pca_100d.explained_variance_ratio_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Eigenvalue scree plot (first 20)
ax1.bar(range(1, 21), pca_100d.explained_variance_ratio_[:20],
        color=COLOURS["signal"], alpha=0.8, edgecolor="white")
ax1.set_xlabel("Component")
ax1.set_ylabel("Variance Explained")
ax1.set_title("Per Component")

# Cumulative (first 20)
ax2.plot(range(1, 21), cumvar[:20], "o-",
         color=COLOURS["signal"], markersize=5, linewidth=2)
ax2.axhline(y=0.95, color=COLOURS["accent"], linestyle="--", alpha=0.7, label="95%")
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Variance Explained")
ax2.set_title("Cumulative")
ax2.legend()
ax2.set_ylim(0, 1.05)

n_95 = np.searchsorted(cumvar, 0.95) + 1

fig.suptitle(f"100D ambient, but {n_95} components capture 95% of variance", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/02b_100d_to_3d.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{n_95} components for 95% of variance — close to the true intrinsic dimension of 3")

## Estimating Intrinsic Dimension

We've been using the "elbow" in the explained variance plot to eyeball intrinsic dimension. Here are two more principled approaches:

1. **95% variance threshold**: count components needed for 95% cumulative variance
2. **Participation ratio**: $d_{\text{eff}} = \frac{(\sum \lambda_i)^2}{\sum \lambda_i^2}$ — a single number summarising how many eigenvalues are "active"

In [ ]:
# Compare two intrinsic dimension estimators

eigenvalues = pca_100d.explained_variance_

# Method 1: 95% variance threshold
cumvar_full = np.cumsum(pca_100d.explained_variance_ratio_)
d_95 = np.searchsorted(cumvar_full, 0.95) + 1

# Method 2: Participation ratio
d_pr = (eigenvalues.sum()) ** 2 / (eigenvalues ** 2).sum()

print("Intrinsic dimension estimates:")
print(f"  True intrinsic dimension:    {Z_true.shape[1]}")
print(f"  95% variance threshold:      {d_95}")
print(f"  Participation ratio:         {d_pr:.1f}")
print(f"  Ambient dimension:           {X_100d.shape[1]}")

# Visualise on the same eigenvalue plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1, 16), eigenvalues[:15], color=COLOURS["signal"], alpha=0.8, edgecolor="white")
ax.axvline(d_95, color=COLOURS["accent"], linewidth=2, linestyle="--",
           label=f"95% threshold: d={d_95}")
ax.axvline(d_pr, color=COLOURS["success"], linewidth=2, linestyle="-.",
           label=f"Participation ratio: d={d_pr:.1f}")
ax.set_xlabel("Component")
ax.set_ylabel("Eigenvalue")
ax.set_title("Two ways to estimate intrinsic dimension")
ax.legend()
plt.tight_layout()
plt.savefig("visuals/02b_dimension_estimates.png", dpi=150, bbox_inches="tight")
plt.show()

Both methods point to the same conclusion: despite having 100 columns, this data has ~3 real dimensions. The ambient dimension is 100. The intrinsic dimension is 3. The gap between them is where dimensionality reduction lives.

<details>
<summary><b>The Maths Behind This</b></summary>

The **participation ratio** is defined as:

$$d_{\text{eff}} = \frac{\left(\sum_{i=1}^{d} \lambda_i\right)^2}{\sum_{i=1}^{d} \lambda_i^2}$$

Intuition: if all $d$ eigenvalues are equal ($\lambda_i = c$), then $d_{\text{eff}} = d$ — all dimensions are active. If one eigenvalue dominates ($\lambda_1 \gg \lambda_i$ for $i > 1$), then $d_{\text{eff}} \approx 1$.

The participation ratio smoothly interpolates between "everything matters" and "one thing matters," giving a continuous estimate of effective dimensionality.

</details>

## Where This Matters

**Model selection:** If your data is intrinsically 5D, training a model in 500D means 495 of those dimensions are pure noise. Every useless dimension is a chance for the model to overfit.

**Computational cost:** Distance computations scale with ambient dimension. Working in intrinsic dimension can speed things up by orders of magnitude.

## Feynman Check

1. **A dataset has 500 features. PCA shows 99% variance in 8 components. What's the intrinsic dimension?**

2. **Explain intrinsic vs ambient dimension using a garden hose analogy.** (Hint: a garden hose is a 1D object — water flows along it — but it coils through 3D space.)

You now understand the gap between what data *looks like* and what it *is*. In **02c — The Manifold Hypothesis**, we formalise this and see why it's the reason dimensionality reduction works at all.